In [24]:
import torch
import torch.nn.functional as F

import librosa

from src.model.age_sex.wavlm_demographics import WavLMWrapper as WavLMWrapperAgeSex
from src.model.voice_quality.wavlm_voice_quality import WavLMWrapper as WavLMWrapperVoiceQuality

In [2]:
# Label list
SEX_LABELS = [
    'female',
    'male',
]
VOICE_QUALITY_LABELS = [
    'shrill', 'nasal', 'deep',  # Pitch
    'silky', 'husky', 'raspy', 'guttural', 'vocal-fry', # Texture
    'booming', 'authoritative', 'loud', 'hushed', 'soft', # Volume
    'crisp', 'slurred', 'lisp', 'stammering', # Clarity
    'singsong', 'pitchy', 'flowing', 'monotone', 'staccato', 'punctuated', 'enunciated', 'hesitant', # Rhythm
]

In [3]:
# Load models
device = "cpu"

age_sex_model = WavLMWrapperAgeSex.from_pretrained("tiantiaf/wavlm-large-age-sex").eval().to(device)
voice_quality_model = WavLMWrapperVoiceQuality.from_pretrained("tiantiaf/wavlm-large-voice-quality").eval().to(device)

In [4]:
# Input data would be audio torch.tensor, mono channel, shape 1 x n_samples
# I'm loading a wav file just for example. In the API, it must accept a list of floats.
wave_path = "/workspace/decoding/assets/misc/LJ037-0171.wav"
wave, _ = librosa.load(wave_path, sr=16_000)
wave = torch.tensor(wave)[None]

wave.shape

torch.Size([1, 121344])

In [27]:
return_value = "prob" # logit, prob, logprob

with torch.no_grad():
    # Predict age and sex
    age_pred, sex_pred = age_sex_model(wave)
    sex_prob = F.softmax(sex_pred, dim=1)
    sex_logprob = F.log_softmax(sex_pred, dim=1)
    age_pred, sex_pred = age_pred.squeeze().item(), sex_pred.squeeze().tolist()
    sex_prob, sex_logprob = sex_prob.squeeze().tolist(), sex_logprob.squeeze().tolist()
    # Predict voice quality
    vq_pred = voice_quality_model(wave, return_feature=False)
    vq_prob = F.sigmoid(vq_pred)
    vq_logprob = vq_prob.log()
    vq_pred, vq_prob, vq_logprob = vq_pred.squeeze().tolist(), vq_prob.squeeze().tolist(), vq_logprob.squeeze().tolist()

if return_value == "logit":
    outputs = {
        **{"age": age_pred},
        **{l: sex_pred[i] for i, l in enumerate(SEX_LABELS)},
        **{l: vq_pred[i] for i, l in enumerate(VOICE_QUALITY_LABELS)},
    }

elif return_value == "prob":
    outputs = {
        **{"age": age_pred},
        **{l: sex_prob[i] for i, l in enumerate(SEX_LABELS)},
        **{l: vq_prob[i] for i, l in enumerate(VOICE_QUALITY_LABELS)},
    }

elif return_value == "logprob":
    outputs = {
        **{"age": age_pred},
        **{l: sex_logprob[i] for i, l in enumerate(SEX_LABELS)},
        **{l: vq_logprob[i] for i, l in enumerate(VOICE_QUALITY_LABELS)},
    }

outputs

/root/miniconda3/envs/vox_profile/lib/python3.8/site-packages/torch/nn/functional.py:5193: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  warnings.warn(


{'age': 0.4008750319480896,
 'female': 0.9999961853027344,
 'male': 3.758110324270092e-06,
 'shrill': 4.250781330483733e-06,
 'nasal': 6.835725798737258e-07,
 'deep': 3.995251245214604e-05,
 'silky': 0.15042513608932495,
 'husky': 0.0010861388873308897,
 'raspy': 1.3505044080375228e-05,
 'guttural': 5.276706372241452e-25,
 'vocal-fry': 3.676038206473775e-12,
 'booming': 0.0004597914812620729,
 'authoritative': 0.49461716413497925,
 'loud': 0.38862818479537964,
 'hushed': 0.0,
 'soft': 0.577318549156189,
 'crisp': 0.979983389377594,
 'slurred': 5.185524400985742e-07,
 'lisp': 0.0,
 'stammering': 1.5683135345945232e-11,
 'singsong': 2.5314015328348205e-09,
 'pitchy': 2.4329673831147147e-09,
 'flowing': 0.30733972787857056,
 'monotone': 9.469781571869387e-34,
 'staccato': 0.0,
 'punctuated': 1.1708659997111681e-07,
 'enunciated': 0.004758759867399931,
 'hesitant': 6.01040774199646e-05}

[4.140652656555176, -8.350937843322754]